In [ ]:
import nltk
from nltk.tag import hmm
from nltk.probability import LidstoneProbDist
from collections import defaultdict, Counter
import random
# Downloads
nltk.download('brown', quiet=True)
nltk.download('universal_tagset', quiet=True)
from nltk.corpus import brown


In [ ]:
tagged_sents = brown.tagged_sents(tagset='universal')[:5000]

# HMM training: states = POS tags, symbols = words
trainer = hmm.HiddenMarkovModelTrainer()
hmm_tagger = trainer.train_supervised(
    tagged_sents,
    estimator=lambda fd, bins: LidstoneProbDist(fd, 0.1, bins)
)

In [ ]:
def predict_next_hmm(word):
    word = word.lower()

    try:
        # Tag the word to get its POS
        current_tag = hmm_tagger.tag([word])[0][1]
    except:
        return None  # word not in model

    # 1️⃣ Sample next POS using the transition probabilities
    next_tag_dist = hmm_tagger._transitions[current_tag]  # ProbDist
    next_tag_probs = [(tag, next_tag_dist.prob(tag)) for tag in next_tag_dist.samples()]

    if not next_tag_probs:
        return None

    # Weighted random choice of next POS
    tags, weights = zip(*next_tag_probs)
    next_tag = random.choices(tags, weights=weights)[0]

    # 2️⃣ Sample next word using emission probabilities of that POS
    emission_dist = hmm_tagger._outputs[next_tag]  # ProbDist
    words, weights = zip(*[(w, emission_dist.prob(w)) for w in emission_dist.samples()])

    next_word = random.choices(words, weights=weights)[0]
    return next_word



In [ ]:
test_words = ["the", "is", "will", "have"]

for w in test_words:
    print(f"{w} → {predict_next_hmm(w)}")

the → international
is → therefore
will → frequently
have → to
